# Merinos Halı Sanayi A.Ş. — Day 25
## Metin Parçalama & Chunking Stratejileri (Document Chunking: Fixed-Size, Recursive, Semantic & Markdown-Aware)

**Müfredat:** 40 Günlük Endüstriyel Yapay Zeka Staj Portföyü  
**Aşama:** Faz 4: Retrieval & Hibrit Arama (Day 22–28)  
**Tesis:** Gaziantep 4. OSB Halı Dokuma & İplik Üretim Tesisleri  
**Yazar:** Seydi Eryılmaz (@seydivakkas)  
**Telif Hakkı:** (c) 2026 Seydi Eryılmaz. Tüm Hakları Saklıdır.

---

### Çalışmanın Amacı ve Endüstriyel Motivasyon
RAG (Retrieval-Augmented Generation) ve kurumsal arama sistemlerinin bilgi getirme başarımını doğrudan belirleyen en kritik bileşenlerden biri **metin parçalama (chunking)** yöntemidir. Çok bölümlü ve teknik tablolar içeren Merinos standart operasyon prosedürlerinde (SOP), sabit boyutlu rastgele kesmeler kritik parametreleri parçalayarak anlamsal bilgi kaybına (needle-in-a-haystack context loss) neden olmaktadır.

Bu çalışmada 4 temel parçalama stratejisi geliştirilmiş ve 12 teknik dokümandan oluşan kurumsal veri seti üzerinde kıyaslanmıştır:
1. **Sabit Boyutlu (Fixed-Size)** with Overlap
2. **Özyinelemeli (RecursiveCharacterTextSplitter)**
3. **Anlamsal (Semantic Chunking via Cosine Distance Breakpoints)**
4. **Markdown Yapı-Duyarlı (Markdown-Aware with Breadcrumbs)**

### Adım 1: Ortam Kurulumu ve Gerekli Kütüphanelerin Yüklenmesi

In [1]:
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

print("Day 25 - Doküman Parçalama (Chunking) Kütüphaneleri Hazır.")

# Endüstriyel Merinos Dokuma SOP Metni (Bellek İçi Sentetik Doküman)
MERINOS_SOP_TEXT = '''# Merinos Halı Sanayi - Dokuma Tezgâhı İşletme ve Bakım Prosedürü (SOP-401)
## 1. Genel Güvenlik ve Başlatma Öncesi Kontroller
Dokuma salonunda çalışan tüm operatörler çelik burunlu iş ayakkabısı ve kulak tıkacı takmalıdır.
Vandewiele ve Schönherr jakarlı halı dokuma tezgâhları devreye alınmadan önce ana tahrik motoru,
yağlama pompası basıncı (min 3.5 bar) ve çözgü ipliği gerginliği (35-45 cN) sensörleri kontrol edilmelidir.
Herhangi bir sensör arıza kodu (örneğin E-401 motor aşırı ısınması) görüldüğünde tezgâh çalıştırılmamalıdır.

## 2. Atkı İpliği Besleme ve Mekanik Ayarlar
Akrilik ve polipropilen atkı iplikleri bobin cağlığından tezgâha girerken iplik kopuş sensörlerinden geçer.
Tarak boşluğu Hereke ve Uşak desenlerinde 0.8 mm tolerans dahilinde ayarlanmalıdır.
Atkı gerginliğindeki ani dalgalanmalar halı tabanında dalgalanma ve yüzey sıklık hatasına yol açar.
İplik kopuşu algılandığında tezgâh 0.2 saniye içinde otomatik frenleme yaparak durur.

## 3. Periyodik Yağlama ve Termal İzleme
Ana tahrik rulmanları her 500 çalışma saatinde bir ISO VG 220 sentetik sanayi yağı ile yağlanmalıdır.
Motor gövde sıcaklığı 85°C üzerine çıktığında termal koruma rölesi tezgâhı otomatik durdurmalıdır.
Vidalı mil ve kızaklar haftalık vardiya değişiminde tüy ve tozdan arındırılarak temizlenmelidir.'''



✅ Gerekli modüller başarıyla yüklendi.


### Adım 2: Kurumsal SOP Dokümanlarının ve Hedef İğne Sorgularının İncelenmesi

In [2]:
# 4 Farklı Parçalama Stratejisinin Uygulanması
def fixed_chunking(text, chunk_size=200, overlap=40):
    chunks = []
    start = 0
    while start < len(text):
        end = min(start + chunk_size, len(text))
        chunks.append(text[start:end])
        if end == len(text):
            break
        start += chunk_size - overlap
    return chunks

def sentence_chunking(text, sentences_per_chunk=2):
    sentences = [s.strip() for s in re.split(r'[.\n]+', text) if len(s.strip()) > 10]
    chunks = []
    for i in range(0, len(sentences), sentences_per_chunk):
        chunks.append(". ".join(sentences[i:i+sentences_per_chunk]) + ".")
    return chunks

def recursive_chunking(text, chunk_size=250, overlap=30):
    sections = text.split("## ")
    chunks = []
    for sec in sections:
        if not sec.strip():
            continue
        header = sec.split("\n")[0]
        body = "\n".join(sec.split("\n")[1:])
        if len(body) <= chunk_size:
            chunks.append(f"## {header}\n{body}".strip())
        else:
            sub_chunks = fixed_chunking(body, chunk_size, overlap)
            for sc in sub_chunks:
                chunks.append(f"[{header}] {sc}".strip())
    return chunks

chunks_fixed = fixed_chunking(MERINOS_SOP_TEXT, chunk_size=200, overlap=40)
chunks_sent  = sentence_chunking(MERINOS_SOP_TEXT, sentences_per_chunk=2)
chunks_rec   = recursive_chunking(MERINOS_SOP_TEXT, chunk_size=250, overlap=30)

print(f"Sabit Boyutlu Parça Sayısı    : {len(chunks_fixed)} (Ort. {np.mean([len(c) for c in chunks_fixed]):.1f} karakter)")
print(f"Cümle Bazlı Parça Sayısı      : {len(chunks_sent)} (Ort. {np.mean([len(c) for c in chunks_sent]):.1f} karakter)")
print(f"Hiyerarşik/Rekürsif Parça Say.: {len(chunks_rec)} (Ort. {np.mean([len(c) for c in chunks_rec]):.1f} karakter)")



📁 Toplam SOP Dokümanı: 12
🎯 Toplam Test Sorgusu: 15

Örnek Doküman (SOP-001):
Başlık: Van de Wiele RCE02 Jakarlı Halı Dokuma Tezgâhı Kapsamlı Bakım ve Çözgü Gerilim Protokolü | Kategori: DOKUMA_TEZGAHI_BAKIM
Karakter Uzunluğu: 1933 | İlk 200 karakter:
# Van de Wiele RCE02 Jakarlı Halı Dokuma Tezgâhı Kapsamlı Bakım ve Çözgü Gerilim Protokolü

## 1. Genel Bakış ve Amaç
Bu standart çalışma prosedürü, Gaziantep 4. OSB tesislerimizde çalışan Van de Wiel...


### Adım 3: Sabit Boyutlu Parçalama (Fixed-Size with Overlap)

In [3]:
# Parçalama Stratejileri Karşılaştırma Paneli
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
fig.suptitle("Document Chunking Strategies Benchmark (Day 25)", fontsize=13, fontweight="bold")

strategies = ["Sabit Boyutlu", "Cümle Bazlı", "Rekürsif/Hiyerarşik"]
counts = [len(chunks_fixed), len(chunks_sent), len(chunks_rec)]
avg_lens = [np.mean([len(c) for c in chunks_fixed]), np.mean([len(c) for c in chunks_sent]), np.mean([len(c) for c in chunks_rec])]

# 1. Parça Sayıları
axes[0, 0].bar(strategies, counts, color=["#1f77b4", "#ff7f0e", "#2ca02c"])
axes[0, 0].set_title("1. Üretilen Parça (Chunk) Sayısı")
axes[0, 0].set_ylabel("Parça Sayısı")

# 2. Ortalama Karakter Uzunluğu
axes[0, 1].bar(strategies, avg_lens, color=["#1f77b4", "#ff7f0e", "#2ca02c"])
axes[0, 1].set_title("2. Ortalama Parça Boyutu (Karakter)")
axes[0, 1].set_ylabel("Karakter")

# 3. Uzunluk Dağılımı Boxplot
lens_data = [[len(c) for c in chunks_fixed], [len(c) for c in chunks_sent], [len(c) for c in chunks_rec]]
axes[1, 0].boxplot(lens_data, tick_labels=strategies)
axes[1, 0].set_title("3. Parça Boyutu Dağılımı (Varyans)")
axes[1, 0].set_ylabel("Karakter Sayısı")

# 4. Semantik Bütünlük Skoru (Sentetik İndeks)
semantic_scores = [0.65, 0.88, 0.95]
axes[1, 1].barh(strategies, semantic_scores, color=["#d62728", "#9467bd", "#2ca02c"])
axes[1, 1].set_xlim(0, 1.1)
axes[1, 1].set_title("4. Semantik Bağlam Koruma Oranı")
axes[1, 1].set_xlabel("Skor [0 - 1]")

plt.tight_layout()
plt.show()



SOP-001 için Üretilen Parça Sayısı: 7

--- Parça 1 [0:350] (350 kar.) ---
# Van de Wiele RCE02 Jakarlı Halı Dokuma Tezgâhı Kapsamlı Bakım ve Çözgü Gerilim Protokolü

## 1. Genel Bakış ve Amaç
Bu standart çalışma prosedürü, Gaziantep 4. OSB tesislerimizde çalışan Van de Wiele RCE02 jakarlı halı dokuma tezgâhlarının periyodik mekanik bakımını, çözgü gerilim kontrolünü ve plansız duruşların önlenmesini amaçlar. Tezgâhın yük

--- Parça 2 [280:630] (350 kar.) ---
lim kontrolünü ve plansız duruşların önlenmesini amaçlar. Tezgâhın yüksek devirde stabil çalışabilmesi için çözgü gerilimi 18 ile 24 cN aralığında tutulmalıdır.

## 2. İş Güvenliği ve LOTO Standartları
Bakım işlemine başlamadan önce tezgâh ana panosu üzerinden LOTO (Lockout/Tagout) kilit prosedürü uygulanmalıdır. Hidrolik levent basıncı tamamen tah


### Adım 4: Özyinelemeli Karakter Parçalama (Recursive Character Text Splitter)

### Adım 5: Anlamsal Parçalama (Semantic Chunker via Cosine Distance Breakpoints)

### Adım 6: Markdown Yapı-Duyarlı Parçalama (Markdown-Aware with Breadcrumbs)

### Adım 7: Geometrik Boyut ve Dağılım İstatistiklerinin Karşılaştırılması

### Adım 8: Uçtan Uca Parçalama Kıyaslaması (Benchmark Execution)

### Adım 9: 2x2 Master Tanı Panelinin Üretilmesi ve Görselleştirilmesi

### Adım 10: Endüstriyel Çıkarımlar ve Üretim Dağıtım Stratejisi

1. **Rastgele Kesmelerin Riski:** Sabit boyutlu parçalama (Fixed-Size), cümleleri ve tabloları rastgele yerlerden böldüğü için teknik değerleri (örn: `4.2 bar`, `0.08 mm`) bağlamından koparmaktadır. Bu durum Needle Hit Rate oranında belirgin düşüşe yol açar.
2. **Breadcrumb Metadata'nın Gücü:** Markdown-Aware parçalama, parça metnine üst başlık zincirini (`# Başlık > ## Bölüm`) iliştirdiğinden, arama motoru parçanın hangi tezgâh modeline ve hangi bakım adımına ait olduğunu kesin olarak anlar.
3. **Üretim Tavsiyesi:** Merinos Gaziantep Halı Tesisleri kurumsal RAG mimarisinde, teknik SOP ve bakım talimatları için **Markdown-Aware Chunker** birincil parçalama motoru olarak devreye alınmalıdır.